# data-cleaning_001

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (101).ipynb`

| Field | Value |
|---|---|
| Section | `data_cleaning` |
| Market | `us_equities` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 3 |
| Detected functions | validate, __init__, _map_side, _retry, get_position, get_account, get_open_orders, cancel_all_orders, submit_market_order, confirm_order_filled, get_latest_quote, get_fee_rate |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
!pip install pandas numpy loguru alpaca-trade-api python-dotenv
!pip install alpaca-py==0.22.0 pandas numpy loguru

In [ ]:
import pandas as pd
import numpy as np
import time
import json
import os
import math
from datetime import datetime, timedelta, timezone
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.trading.client import TradingClient
from alpaca.data.historical import CryptoHistoricalDataClient
from alpaca.trading.requests import MarketOrderRequest, GetOrdersRequest
from alpaca.trading.enums import OrderStatus, QueryOrderStatus
from loguru import logger
from google.colab import drive
from google.colab import userdata

# Mount Google Drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/My Drive/crypto_trader/'
os.makedirs(DRIVE_PATH, exist_ok=True)

# ===================== Configuration =====================
class TradingConfig:
    try:
        API_KEY = userdata.get('ALPACA_API_KEY')
        API_SECRET = userdata.get('ALPACA_SECRET_KEY')
    except:
        API_KEY = os.environ.get('ALPACA_API_KEY', input("Enter ALPACA_API_KEY: "))
        API_SECRET = os.environ.get('ALPACA_SECRET_KEY', input("Enter ALPACA_SECRET_KEY: "))
    BASE_URL = "https://paper-api.alpaca.markets"
    SYMBOL = "SOL/USD"
    CAPITAL = 100_000
    TAKE_PROFIT_PCT = 0.015
    STOP_LOSS_PCT = 0.005
    ENTRY_STOP_LOSS_PCT = 0.03
    BARS_LOOKBACK = 3000
    POSITION_STATE_FILE = os.path.join(DRIVE_PATH, "position_state_5min.json")
    LOG_FILE = os.path.join(DRIVE_PATH, "crypto_trader_5min.log")
    GRACE_SECONDS = 2
    ORDER_CONFIRM_TIMEOUT = 5
    ORDER_RETRY_DELAY = 2
    MIN_PRICE = 0.01
    MIN_STD = 1e-6
    PRICE_DECIMALS = 2
    FEE_RATE = 0.0025

    @staticmethod
    def validate():
        if not (TradingConfig.API_KEY and TradingConfig.API_SECRET):
            raise ValueError("API credentials not set.")

# ===================== Logger Setup =====================
logger.remove()
logger.add(
    TradingConfig.LOG_FILE,
    rotation="10 MB",
    level="DEBUG",
    format="{time:YYYY-MM-DD HH:mm:ss} [{level}] {message}"
)
logger.add(
    sink=lambda msg: print(msg, end=""),
    level="INFO",
    format="{time:YYYY-MM-DD HH:mm:ss} [{level}] {message}"
)

# ===================== Custom Exceptions =====================
class WashTradeError(Exception):
    pass

# ===================== Alpaca API =====================
class AlpacaClient:
    def __init__(
        self,
        trading_client=None,
        data_client=None,
        retry_defaults=None,
    ):
        TradingConfig.validate()
        self.config = TradingConfig
        self.data_client = data_client or CryptoHistoricalDataClient(
            self.config.API_KEY, self.config.API_SECRET
        )
        self.trading_client = trading_client or TradingClient(
            self.config.API_KEY, self.config.API_SECRET, paper=True
        )
        self.retry_defaults = retry_defaults or dict(max_attempts=3, backoff_base=0.3, backoff_factor=2.0)

    def _map_side(self, side: str) -> OrderSide:
        side_norm = (side or "").strip().lower()
        if side_norm in ("buy", "b", "long"):
            return OrderSide.BUY
        elif side_norm in ("sell", "s", "short"):
            return OrderSide.SELL
        raise ValueError(f"Unknown side: {side!r}. Use 'buy' or 'sell'.")

    def _retry(self, fn, *args, **kwargs):
        params = {
            "max_attempts": kwargs.pop("max_attempts", self.retry_defaults["max_attempts"]),
            "backoff_base": kwargs.pop("backoff_base", self.retry_defaults["backoff_base"]),
            "backoff_factor": kwargs.pop("backoff_factor", self.retry_defaults["backoff_factor"]),
        }
        attempt, last_exc = 0, None
        while attempt < params["max_attempts"]:
            attempt += 1
            try:
                return fn(*args, **kwargs)
            except Exception as exc:
                last_exc = exc
                wait = params["backoff_base"] * (params["backoff_factor"] ** (attempt - 1))
                logger.debug(
                    f"[{self.config.SYMBOL}] attempt {attempt}/{params['max_attempts']} failed: "
                    f"{type(exc).__name__} - {exc}; retrying in {wait:.2f}s",
                    exc_info=True
                )
                time.sleep(wait)
        logger.error(f"[{self.config.SYMBOL}] All {params['max_attempts']} attempts failed for {fn.__name__}")
        raise last_exc

    def get_position(self, symbol: str):
        try:
            pos = self.trading_client.get_open_position(symbol)
            return True, pos
        except Exception as exc:
            logger.debug(f"[{symbol}] get_position failed: {type(exc).__name__}: {exc}", exc_info=True)
            return False, None

    def get_account(self):
        try:
            return self.trading_client.get_account()
        except Exception as e:
            logger.error(f"[{self.config.SYMBOL}] Failed to fetch account details: {type(e).__name__}: {e}", exc_info=True)
            return None

    def get_open_orders(self, symbol: str):
        try:
            orders = self.trading_client.get_orders(
                GetOrdersRequest(status=QueryOrderStatus.OPEN, symbols=[symbol])
            )
            return list(orders or [])
        except Exception as e:
            logger.error(f"[{symbol}] Failed to fetch open orders: {type(e).__name__}: {e}", exc_info=True)
            return []

    def cancel_all_orders(self, symbol: str):
        try:
            for order in self.get_open_orders(symbol):
                try:
                    self.trading_client.cancel_order_by_id(order.id)
                    logger.info(f"[{symbol}] Canceled order {order.id}")
                except Exception as inner:
                    logger.warning(f"[{symbol}] Failed to cancel order {order.id}: {type(inner).__name__}: {inner}", exc_info=True)
            return True
        except Exception as e:
            logger.error(f"[{symbol}] Failed to cancel orders: {type(e).__name__}: {e}", exc_info=True)
            return False

    def submit_market_order(self, symbol: str, qty, side: str):
        try:
            order = MarketOrderRequest(
                symbol=symbol,
                qty=qty,
                side=self._map_side(side),
                time_in_force="gtc",
            )
            submitted_order = self._retry(self.trading_client.submit_order, order_data=order)
            logger.info(f"[{symbol}] Market {side} order placed: qty={qty}, id={getattr(submitted_order,'id',None)}")
            return submitted_order
        except Exception as e:
            response = getattr(e, 'response', {}) or {}
            error_detail = f"Status: {response.get('status_code', 'N/A')}, Message: {response.get('message', response.get('detail', str(e)))}" if isinstance(response, dict) else str(e)
            if "wash trade" in error_detail.lower():
                logger.warning(f"[{symbol}] Wash trade detected, retrying after {self.config.ORDER_RETRY_DELAY}s")
                time.sleep(self.config.ORDER_RETRY_DELAY)
                self.cancel_all_orders(symbol)
                raise WashTradeError
            logger.error(f"[{symbol}] Market order failed: {type(e).__name__}: {error_detail}, Response: {response}")
            raise

    def confirm_order_filled(self, symbol: str, order_id: str, timeout=None):
        timeout = timeout or self.config.ORDER_CONFIRM_TIMEOUT
        start_time = time.time()
        while time.time() - start_time < timeout:
            try:
                order = self.trading_client.get_order_by_id(order_id)
                status = str(getattr(order, "status", "")).lower()
                if "filled" in status:
                    logger.info(f"[{symbol}] Order {order_id} filled")
                    return True
                if any(x in status for x in ("canceled", "cancelled", "rejected")):
                    logger.warning(f"[{symbol}] Order {order_id} {status}")
                    return False
            except Exception as e:
                logger.error(f"[{symbol}] Failed to check order {order_id} status: {type(e).__name__}: {e}", exc_info=True)
            time.sleep(0.5)
        logger.warning(f"[{symbol}] Order {order_id} confirmation timed out after {timeout}s")
        return False

    def get_latest_quote(self, symbol: str):
        try:
            from alpaca.data.requests import CryptoLatestQuoteRequest
            req = CryptoLatestQuoteRequest(symbol_or_symbols=[symbol])
            for _ in range(3):
                quote_resp = self.data_client.get_crypto_latest_quote(req)
                q = quote_resp.get(symbol) if quote_resp else None
                if not q:
                    logger.warning(f"[{symbol}] No valid quote, retrying...")
                    time.sleep(0.5)
                    continue
                bid, ask = float(q.bid_price or 0.0), float(q.ask_price or 0.0)
                if bid < self.config.MIN_PRICE or ask < self.config.MIN_PRICE or math.isclose(bid, 0.0) or math.isclose(ask, 0.0):
                    logger.warning(f"[{symbol}] Invalid quote: bid={bid}, ask={ask}")
                    time.sleep(0.5)
                    continue
                return bid, ask
            return 0.0, 0.0
        except Exception as e:
            logger.warning(f"[{symbol}] Quote fetch failed: {type(e).__name__}: {e}", exc_info=True)
            return 0.0, 0.0

    def get_fee_rate(self):
        return float(self.config.FEE_RATE)

# ===================== State Management =====================
class PositionState:
    @staticmethod
    def load():
        if os.path.exists(TradingConfig.POSITION_STATE_FILE):
            try:
                with open(TradingConfig.POSITION_STATE_FILE, "r") as f:
                    state = json.load(f)
                    if not isinstance(state, dict):
                        raise ValueError("Invalid state format")
                    return state
            except Exception as e:
                logger.warning(f"Position state corrupted: type={type(e).__name__}, message={str(e)}. Starting fresh.", exc_info=True)
        return {}

    @staticmethod
    def save(state):
        tmp_file = TradingConfig.POSITION_STATE_FILE + ".tmp"
        try:
            with open(tmp_file, "w") as f:
                json.dump(state, f, indent=2)
            os.replace(tmp_file, TradingConfig.POSITION_STATE_FILE)
        except Exception as e:
            logger.error(f"Failed to save position state: type={type(e).__name__}, message={str(e)}", exc_info=True)
            raise

    @staticmethod
    def clear(symbol):
        state = PositionState.load()
        if symbol in state:
            del state[symbol]
            try:
                PositionState.save(state)
                logger.info(f"[{symbol}] Position state cleared.")
            except Exception as e:
                logger.error(f"[{symbol}] Failed to clear position state: {type(e).__name__}: {str(e)}", exc_info=True)

    @staticmethod
    def update(symbol, data):
        state = PositionState.load()
        state[symbol] = data
        try:
            PositionState.save(state)
        except Exception as e:
            logger.error(f"[{symbol}] Failed to update position state: {type(e).__name__}: {str(e)}", exc_info=True)

# ===================== Data Fetching =====================
def fetch_bars(client, symbol, bars=TradingConfig.BARS_LOOKBACK):
    from alpaca.data.requests import CryptoBarsRequest
    MAX_ATTEMPTS = 3
    MIN_BARS = 60
    TARGET_BARS = bars
    LOOKBACK_DAYS = 60
    SLEEP_BETWEEN_CALLS = 1
    SLEEP_ON_FAIL = 30
    TIMEFRAME = TimeFrame(5, TimeFrameUnit.Minute)

    collected_bars = pd.DataFrame()
    end_time = datetime.now(timezone.utc)
    min_timestamp = end_time - timedelta(days=LOOKBACK_DAYS)

    for attempt in range(MAX_ATTEMPTS):
        try:
            while len(collected_bars) < TARGET_BARS:
                limit = min(TARGET_BARS - len(collected_bars), 1000)
                request = CryptoBarsRequest(
                    symbol_or_symbols=symbol,
                    timeframe=TIMEFRAME,
                    limit=limit,
                    end=end_time
                )
                logger.debug(f"[{symbol}] Requesting {limit} bars ending {end_time.isoformat()}")
                time.sleep(SLEEP_BETWEEN_CALLS)
                df = client.data_client.get_crypto_bars(request).df
                logger.debug(f"[{symbol}] Fetched rows={len(df)}, cols={df.columns.tolist()}")
                if df.empty:
                    logger.error(f"[{symbol}] No data returned.")
                    break
                if isinstance(df.index, pd.MultiIndex):
                    df = df.reset_index()
                if 'timestamp' not in df.columns and 'time' in df.columns:
                    df.rename(columns={'time': 'timestamp'}, inplace=True)
                required_cols = ["open", "high", "low", "close", "timestamp"]
                missing = [c for c in required_cols if c not in df.columns]
                if missing:
                    logger.error(f"[{symbol}] Missing columns: {missing}")
                    break
                df = df[required_cols].rename(columns={"timestamp": "time"})
                df["time"] = pd.to_datetime(df["time"]).dt.tz_convert(timezone.utc)
                df.drop_duplicates(subset="time", inplace=True)
                df.sort_values("time", inplace=True)
                collected_bars = pd.concat([df, collected_bars], ignore_index=True)
                collected_bars.drop_duplicates(subset="time", inplace=True)
                collected_bars.sort_values("time", inplace=True)
                logger.info(f"[{symbol}] Added {len(df)} bars, total: {len(collected_bars)}")
                if len(df) < limit:
                    logger.info(f"[{symbol}] Less than limit ({limit}) received. Stopping.")
                    break
                if collected_bars["time"].min() <= min_timestamp:
                    logger.info(f"[{symbol}] Reached minimum timestamp {min_timestamp.isoformat()}.")
                    break
                end_time = collected_bars["time"].min() - timedelta(minutes=5)
            if len(collected_bars) < MIN_BARS:
                logger.warning(f"[{symbol}] Only {len(collected_bars)} bars fetched, need at least {MIN_BARS}.")
                if attempt < MAX_ATTEMPTS - 1:
                    logger.info(f"[{symbol}] Retrying in {SLEEP_ON_FAIL} seconds...")
                    time.sleep(SLEEP_ON_FAIL)
                    collected_bars = pd.DataFrame()
                    end_time = datetime.now(timezone.utc)
                    continue
                return None
            logger.info(f"[{symbol}] Successfully fetched {len(collected_bars)} bars.")
            return collected_bars.tail(TARGET_BARS).reset_index(drop=True)
        except Exception as e:
            logger.error(f"[{symbol}] Attempt {attempt+1}/{MAX_ATTEMPTS} failed: {e}")
            if attempt < MAX_ATTEMPTS - 1:
                logger.info(f"[{symbol}] Retrying in {SLEEP_ON_FAIL} seconds...")
                time.sleep(SLEEP_ON_FAIL)
                collected_bars = pd.DataFrame()
                end_time = datetime.now(timezone.utc)
            else:
                return None

# ===================== Indicator Calculation =====================
def compute_jma_signal(
    df: pd.DataFrame,
    smooth_length: int = 14,
    norm_period: int = 60,
    upper_threshold: float = 0.5,
    lower_threshold: float = -0.5,
    rs_period: int = 14,
    rs_percentile: float = 10.0,
    min_price: float = 1e-12,
    min_std: float = 1e-8,
    symbol: str = "SYMBOL",
    logger=None
) -> pd.DataFrame:
    log = logger.info if logger else print
    warn = logger.warning if logger else print
    error = logger.error if logger else print
    debug = logger.debug if logger else print
    try:
        required_cols = ["open", "high", "low", "close", "time"]
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            error(f"[{symbol}] Missing required columns: {missing}")
            return None
        if len(df) < norm_period:
            error(f"[{symbol}] Insufficient data: {len(df)} bars, need at least {norm_period}")
            return None
        if not pd.api.types.is_datetime64_any_dtype(df["time"]):
            error(f"[{symbol}] 'time' column is not datetime")
            return None
        def _clean_price_series(s: pd.Series) -> pd.Series:
            s = s.replace(0, np.nan).astype(float)
            s = s.interpolate(method="linear", limit_direction="both")
            return s.ffill().bfill()
        for col in ["open", "high", "low", "close"]:
            df[col] = _clean_price_series(df[col])
            if df[col].isna().any():
                warn(f"[{symbol}] {col} still contains NaN after cleaning.")
        btc_vol_threshold = 1e-7 * df["close"].mean()
        close_diff_early = df["close"].head(2).diff().dropna()
        if len(close_diff_early) >= 1 and close_diff_early.abs().mean() < btc_vol_threshold:
            warn(f"[{symbol}] Early close prices near-constant "
                 f"(mean diff: {close_diff_early.abs().mean():.8f}, threshold: {btc_vol_threshold:.8f}). Neutral signal.")
            df["signal"] = 0
            return df
        close_diff = df["close"].diff().dropna()
        if len(close_diff) >= norm_period and close_diff.abs().mean() < btc_vol_threshold:
            warn(f"[{symbol}] Close prices near-constant "
                 f"(mean diff: {close_diff.abs().mean():.8f}, threshold: {btc_vol_threshold:.8f}). Neutral signal.")
            df["signal"] = 0
            return df
        price = df["close"].astype(float)
        alpha = 2.0 / (smooth_length + 1.0)
        jma = price.ewm(alpha=alpha, adjust=False).mean()
        df["jma"] = jma
        roll_mean = jma.rolling(window=norm_period, min_periods=1).mean().bfill().interpolate(method="linear")
        roll_std = jma.rolling(window=norm_period, min_periods=1).std(ddof=0).bfill().interpolate(method="linear").clip(lower=min_std)
        df["jma_roll_mean"] = roll_mean
        df["jma_roll_std"] = roll_std
        norm_jma = (jma - roll_mean) / roll_std
        norm_jma = norm_jma.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        df["norm_jma"] = norm_jma
        min_price = max(min_price, 1e-12)
        o = df["open"].clip(lower=min_price)
        h = df["high"].clip(lower=min_price)
        l = df["low"].clip(lower=min_price)
        c = df["close"].clip(lower=min_price)
        log_hc = np.log(h / c)
        log_ho = np.log(h / o)
        log_lc = np.log(l / c)
        log_lo = np.log(l / o)
        rs_combined = (log_hc * log_ho) + (log_lc * log_lo)
        rs_vol = rs_combined.rolling(window=rs_period, min_periods=1).mean() * 1e6
        df["rs_vol"] = rs_vol
        min_vol_threshold = 1e-4
        rs_threshold = rs_vol.rolling(window=norm_period, min_periods=1).quantile(rs_percentile / 100.0)
        rs_threshold = rs_threshold.clip(lower=min_vol_threshold)
        df["rs_threshold"] = rs_threshold
        long_mask = (norm_jma > upper_threshold) & (rs_vol > rs_threshold)
        short_mask = (norm_jma < lower_threshold) & (rs_vol > rs_threshold)
        signals = pd.Series(0, index=df.index, dtype=np.int8)
        signals[long_mask] = 1
        signals[short_mask] = 2
        df["signal"] = signals
        if len(df) >= 3 and df.iloc[-2]["signal"] != 0:
            triggered = int(df.iloc[-2]["signal"])
            trigger_time = df.iloc[-2]["time"]
            trigger_bar = df.iloc[-3]["time"]
            debug(
                f"[{symbol}] Signal {triggered} ({'LONG' if triggered == 1 else 'SHORT'}) "
                f"triggered for closed bar {trigger_time}, "
                f"based on trigger bar {trigger_bar}: "
                f"norm_jma={df['norm_jma'].iloc[-3]:.4f}, "
                f"rs_vol={df['rs_vol'].iloc[-3]:.4f}, "
                f"rs_threshold={df['rs_threshold'].iloc[-3]:.4f}"
            )
        return df
    except Exception as exc:
        logger.error(f"[{TradingConfig.SYMBOL}] Indicator computation failed: type={type(exc).__name__}, message={str(exc)}", exc_info=True)
        return None

# ===================== Trading Logic =====================
class CryptoTrader:
    def __init__(self):
        self.client = AlpacaClient()
        self.fee_rate = self.client.get_fee_rate()
        self.max_retries = 3
        logger.info(f"Initialized CryptoTrader for {TradingConfig.SYMBOL} (paper trading, 5-minute timeframe)")

    def sync_position(self, symbol):
        has_pos, pos = self.client.get_position(symbol)
        state = PositionState.load()
        open_orders = self.client.get_open_orders(symbol)
        if open_orders:
            logger.warning(f"[{symbol}] Found {len(open_orders)} open orders, canceling to sync state")
            self.client.cancel_all_orders(symbol)
        if has_pos and symbol not in state:
            logger.warning(f"[{symbol}] Broker position found but no local state. Rebuilding.")
            try:
                qty = float(pos.qty)
                side = 1 if pos.side.lower() == "long" else -1
                entry_price = float(pos.avg_entry_price)
                if entry_price < TradingConfig.MIN_PRICE:
                    logger.error(f"[{symbol}] Invalid entry price: {entry_price}")
                    return
                state[symbol] = {
                    "entry_time": datetime.now(timezone.utc).isoformat(),
                    "entry_price": entry_price,
                    "quantity": qty,
                    "direction": side,
                    "stop_loss_price": entry_price * (1 - TradingConfig.ENTRY_STOP_LOSS_PCT * side),
                    "best_high": entry_price if side == 1 else None,
                    "best_low": entry_price if side == -1 else None,
                    "last_bar_time": datetime.now(timezone.utc).isoformat(),
                    "is_entry_candle": False,
                    "market_order_id": None
                }
                PositionState.update(symbol, state[symbol])
            except Exception as e:
                logger.error(f"[{symbol}] Failed to rebuild position state: {type(e).__name__}: {str(e)}", exc_info=True)
        elif not has_pos and symbol in state:
            logger.warning(f"[{symbol}] Local state exists but no broker position. Clearing.")
            PositionState.clear(symbol)

    def estimate_slippage(self, symbol):
        bid, ask = self.client.get_latest_quote(symbol)
        if bid > 0 and ask > 0:
            mid = (bid + ask) / 2.0
            slippage = (ask - bid) / mid
            logger.debug(f"[{symbol}] Estimated slippage: {slippage:.4%}")
            return slippage
        logger.warning(f"[{symbol}] Using conservative default slippage rate of 1% due to quote failure.")
        return 0.01

    def calculate_quantity(self, symbol, entry_price):
        account = self.client.get_account()
        if account is None:
            logger.error(f"[{symbol}] Cannot calculate quantity: failed to fetch account details")
            return 0.0
        try:
            available_cash = float(account.cash)
            fee_buffer = 1 - self.fee_rate * 2
            qty = (available_cash * 0.10 * fee_buffer) / entry_price
            return max(round(qty, 8), 0.000001)
        except Exception as e:
            logger.error(f"[{symbol}] Quantity calculation failed: {type(e).__name__}: {str(e)}", exc_info=True)
            return 0.0

    def enter_position(self, symbol, direction, bar, slippage_rate):
        try:
            open_price = float(bar["open"])
            timestamp = bar["time"]
            if open_price < TradingConfig.MIN_PRICE:
                logger.error(f"[{symbol}] Invalid bar price: open={open_price}")
                return
            self.client.cancel_all_orders(symbol)
            entry_price = open_price * (1 + slippage_rate if direction == 1 else 1 - slippage_rate)
            qty = self.calculate_quantity(symbol, entry_price)
            if qty <= 0.000001:
                logger.error(f"[{symbol}] Calculated quantity too low: {qty}")
                return
            entry_side = "buy" if direction == 1 else "sell"
            for attempt in range(self.max_retries):
                try:
                    market_order = self.client.submit_market_order(symbol, qty, entry_side)
                    if not self.client.confirm_order_filled(symbol, market_order.id):
                        logger.warning(f"[{symbol}] Market order {market_order.id} not filled, retrying...")
                        self.client.cancel_all_orders(symbol)
                        continue
                    logger.info(f"[{symbol}] Entered {'LONG' if direction == 1 else 'SHORT'} at ~{entry_price:.4f}, qty={qty}")
                    state = {
                        "entry_time": timestamp.isoformat(),
                        "entry_price": entry_price,
                        "quantity": qty,
                        "direction": direction,
                        "stop_loss_price": entry_price * (1 - TradingConfig.ENTRY_STOP_LOSS_PCT * direction),
                        "best_high": entry_price if direction == 1 else None,
                        "best_low": entry_price if direction == -1 else None,
                        "last_bar_time": timestamp.isoformat(),
                        "is_entry_candle": True,
                        "market_order_id": str(market_order.id)
                    }
                    PositionState.update(symbol, state)
                    break
                except WashTradeError:
                    if attempt == self.max_retries - 1:
                        logger.error(f"[{symbol}] Max retries reached for wash trade")
                        return
                    continue
                except Exception as e:
                    logger.error(f"[{symbol}] Entry attempt {attempt+1} failed: {type(e).__name__}: {str(e)}", exc_info=True)
                    self.client.cancel_all_orders(symbol)
                    if attempt == self.max_retries - 1:
                        logger.error(f"[{symbol}] Max retries reached for entry")
                        return
        except Exception as e:
            logger.error(f"[{symbol}] Enter position failed: {type(e).__name__}: {str(e)}", exc_info=True)

    def should_update_stop(self, symbol, bar, state):
        direction = int(state["direction"])
        high = float(bar["high"])
        low = float(bar["low"])
        if direction == 1:
            best_high = float(state.get("best_high", high))
            return high > best_high
        else:
            best_low = float(state.get("best_low", low))
            return low < best_low

    def update_trailing(self, symbol, bar, state):
        direction = int(state["direction"])
        high = float(bar["high"])
        low = float(bar["low"])
        qty = float(state["quantity"])
        exit_side = "sell" if direction == 1 else "buy"
        if direction == 1:
            best_high = float(state.get("best_high", high))
            if high > best_high:
                best_high = high
                state["best_high"] = best_high
                new_stop_price = best_high * (1 - TradingConfig.STOP_LOSS_PCT)
                current_price = float(bar["close"])
                if current_price <= new_stop_price:
                    self.exit_position(symbol, bar, self.estimate_slippage(symbol), reason="virtual stop hit")
                else:
                    state["stop_loss_price"] = new_stop_price
        else:
            best_low = float(state.get("best_low", low))
            if low < best_low:
                best_low = low
                state["best_low"] = best_low
                new_stop_price = best_low * (1 + TradingConfig.STOP_LOSS_PCT)
                current_price = float(bar["close"])
                if current_price >= new_stop_price:
                    self.exit_position(symbol, bar, self.estimate_slippage(symbol), reason="virtual stop hit")
                else:
                    state["stop_loss_price"] = new_stop_price
        PositionState.update(symbol, state)

    def manage_position(self, symbol, closed_bar, new_bar, slippage_rate):
        state = PositionState.load().get(symbol)
        if not state:
            return False
        try:
            direction = int(state["direction"])
            timestamp = closed_bar["time"]
            if state.get("is_entry_candle", False):
                state["is_entry_candle"] = False
                state["last_bar_time"] = timestamp.isoformat()
                PositionState.update(symbol, state)
                return True
            if self.should_update_stop(symbol, closed_bar, state):
                self.update_trailing(symbol, closed_bar, state)
                logger.info(f"[{symbol}] Updated trailing stop at time={timestamp}")
            else:
                logger.debug(f"[{symbol}] No favorable stop movement at time={timestamp}")
            state["last_bar_time"] = timestamp.isoformat()
            PositionState.update(symbol, state)
            return True
        except Exception as e:
            logger.error(f"[{symbol}] Manage position failed: {type(e).__name__}: {str(e)}", exc_info=True)
            return True

    def exit_position(self, symbol, bar, slippage_rate, reason=""):
        state = PositionState.load().get(symbol)
        if not state:
            return
        has_pos, _ = self.client.get_position(symbol)
        if not has_pos:
            logger.warning(f"[{symbol}] No position found for exit. Clearing state.")
            PositionState.clear(symbol)
            return
        try:
            direction = int(state["direction"])
            qty = float(state["quantity"])
            entry_price = float(state["entry_price"])
            open_price = float(bar["open"])
            timestamp = bar["time"]
            if open_price < TradingConfig.MIN_PRICE:
                logger.error(f"[{symbol}] Invalid open price for exit: {open_price}, time={timestamp}")
                return
            self.client.cancel_all_orders(symbol)
            exit_side = "sell" if direction == 1 else "buy"
            exit_price = open_price * (1 - slippage_rate if direction == 1 else 1 + slippage_rate)
            for attempt in range(self.max_retries):
                try:
                    order = self.client.submit_market_order(symbol, qty, exit_side)
                    if self.client.confirm_order_filled(symbol, order.id):
                        pnl = (exit_price - entry_price) * qty * direction
                        fees = self.fee_rate * (entry_price + exit_price) * qty
                        net_pnl = pnl - fees
                        logger.info(f"[{symbol}] Exited at ~{exit_price:.4f} due to {reason}, Net PnL={net_pnl:.2f}, time={timestamp}")
                        PositionState.clear(symbol)
                        return
                    else:
                        logger.warning(f"[{symbol}] Exit order {order.id} not filled, retrying...")
                        self.client.cancel_all_orders(symbol)
                except WashTradeError:
                    if attempt == self.max_retries - 1:
                        logger.error(f"[{symbol}] Max retries reached for wash trade on exit")
                        return
                    continue
                except Exception as e:
                    logger.error(f"[{symbol}] Exit attempt {attempt+1} failed: {type(e).__name__}: {str(e)}", exc_info=True)
                    self.client.cancel_all_orders(symbol)
                    if attempt == self.max_retries - 1:
                        logger.error(f"[{symbol}] Max exit retries reached.")
                        return
        except Exception as e:
            logger.error(f"[{symbol}] Exit position failed: {type(e).__name__}: {str(e)}", exc_info=True)

    def wait_for_next_bar(self):
        try:
            now = datetime.now(timezone.utc)
            minutes = (now.minute // 5 + 1) * 5
            next_bar = (now.replace(minute=0, second=0, microsecond=0) + timedelta(minutes=minutes)).replace(
                second=TradingConfig.GRACE_SECONDS
            )
            sleep_seconds = max((next_bar - now).total_seconds(), 0)
            if sleep_seconds <= 0:
                sleep_seconds += 300
            logger.info(f"Waiting {sleep_seconds:.1f}s until {next_bar.isoformat()}")
            time.sleep(sleep_seconds)
        except Exception as e:
            logger.error(f"[{TradingConfig.SYMBOL}] Wait for next bar failed: {type(e).__name__}: {str(e)}", exc_info=True)
            time.sleep(300)

    def run(self, max_cycles=float('inf')):
        logger.info(f"Starting CryptoTrader for {TradingConfig.SYMBOL} (paper trading, 5-minute timeframe)")
        error_count = 0
        max_errors = 5
        cycle_count = 0
        last_rate_limit_time = None
        while cycle_count < max_cycles:
            try:
                if last_rate_limit_time and (datetime.now(timezone.utc) - last_rate_limit_time).total_seconds() < 60:
                    logger.warning(f"[{TradingConfig.SYMBOL}] Rate limit detected recently, waiting 60s")
                    time.sleep(60)
                self.wait_for_next_bar()
                self.sync_position(TradingConfig.SYMBOL)
                df = fetch_bars(self.client, TradingConfig.SYMBOL, bars=TradingConfig.BARS_LOOKBACK)
                if df is None or df.shape[0] < 60:
                    logger.warning(f"[{TradingConfig.SYMBOL}] Insufficient or invalid data: {len(df) if df is not None else 0} bars")
                    error_count += 1
                    time.sleep(10)
                    continue
                slippage_rate = self.estimate_slippage(TradingConfig.SYMBOL)
                state = PositionState.load().get(TradingConfig.SYMBOL)
                position_open = bool(state)
                if position_open:
                    self.manage_position(TradingConfig.SYMBOL, df.iloc[-2], df.iloc[-1], slippage_rate)
                else:
                    df = compute_jma_signal(df)
                    if df is None:
                        logger.warning(f"[{TradingConfig.SYMBOL}] Indicator computation failed.")
                        error_count += 1
                        time.sleep(10)
                        continue
                    closed_bar = df.iloc[-2]
                    new_bar = df.iloc[-1]
                    signal = int(closed_bar["signal"])
                    logger.info(f"[{TradingConfig.SYMBOL}] Checking signal: {signal} ({'LONG' if signal == 1 else 'SHORT' if signal == 2 else 'NEUTRAL'}) at {closed_bar['time']}")
                    if signal in (1, 2):
                        direction = 1 if signal == 1 else -1
                        self.enter_position(
                            TradingConfig.SYMBOL, direction, new_bar, slippage_rate
                        )
                logger.info(f"Cycle {cycle_count+1} complete at {datetime.now(timezone.utc).isoformat()}")
                error_count = 0
                cycle_count += 1
            except KeyboardInterrupt:
                logger.info("Trading bot stopped by user.")
                self.client.cancel_all_orders(TradingConfig.SYMBOL)
                break
            except Exception as e:
                error_count += 1
                if "rate limit" in str(e).lower():
                    last_rate_limit_time = datetime.now(timezone.utc)
                    logger.warning(f"[{TradingConfig.SYMBOL}] Rate limit hit, waiting 60s")
                    time.sleep(60)
                    continue
                logger.error(f"Cycle error {error_count}/{max_errors}: type={type(e).__name__}, message={str(e)}", exc_info=True)
                if error_count >= max_errors:
                    logger.error("Max errors reached. Stopping bot.")
                    self.client.cancel_all_orders(TradingConfig.SYMBOL)
                    break
                time.sleep(60)
        logger.info(f"Completed {cycle_count} cycles. Stopping bot.")
        self.client.cancel_all_orders(TradingConfig.SYMBOL)

# ===================== Entry Point =====================
if __name__ == "__main__":
    try:
        trader = CryptoTrader()
        trader.run()
    except Exception as e:
        logger.error(f"Failed to start trader: {type(e).__name__}: {str(e)}", exc_info=True)
        raise